<a href="https://colab.research.google.com/github/Eduardoamaral23/Datathon-Passos-Magicos/blob/main/Modelo_Preditivo_de_Risco_de_Defasagem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo Preditivo de Risco de Defasagem

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, f1_score, precision_score, recall_score
)
import pickle
import warnings
warnings.filterwarnings('ignore')


print("TREINAMENTO DO MODELO PREDITIVO DE RISCO DE DEFASAGEM")


TREINAMENTO DO MODELO PREDITIVO DE RISCO DE DEFASAGEM


# Carregar dados

In [52]:
excel_file_path = "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"
dfs = {}
for year in [2022, 2023, 2024]:
    dfs[year] = pd.read_excel(excel_file_path, sheet_name=f'PEDE{year}')

all_data = pd.concat(dfs.values(), ignore_index=True)

# Padronizar nomes de colunas

In [53]:
for col in all_data.columns:
    if 'INDE' in col and all_data[col].dtype != 'object':
        all_data.rename(columns={col: 'INDE'}, inplace=True)
        break

# Converter para numérico

In [54]:
numeric_cols = ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPV', 'IAN', 'Mat', 'Por', 'Ing', 'Idade', 'IPP']
for col in numeric_cols:
    if col in all_data.columns:
        all_data[col] = pd.to_numeric(all_data[col], errors='coerce')

print(f"\nDados carregados: {len(all_data)} registros")


print("\nRealizando Feature Engineering:")

ml_data = all_data[['IDA', 'IEG', 'IPS', 'IPP', 'IAA', 'IPV', 'IAN', 'INDE', 'Idade', 'Gênero']].copy()
ml_data = ml_data.dropna(subset=['IAN', 'IDA', 'IEG', 'IPS', 'IPP', 'IAA', 'IPV'])


Dados carregados: 3030 registros

Realizando Feature Engineering:


# Variável alvo: Risco de Defasagem (IAN > 5)

In [55]:
ml_data['Risco_Defasagem'] = (ml_data['IAN'] > 5).astype(int)

# Features adicionais

In [56]:
ml_data['Saude_Academica'] = (ml_data['IDA'] + ml_data['IEG']) / 2
ml_data['Bem_Estar_Psico'] = (ml_data['IPS'] + ml_data['IAA']) / 2
ml_data['Risco_Composto'] = ((ml_data['IDA'] < ml_data['IDA'].quantile(0.33)).astype(int) +
                              (ml_data['IEG'] < ml_data['IEG'].quantile(0.33)).astype(int))
ml_data['Gap_Expectativa_Realidade'] = ml_data['IPV'] - ml_data['IDA']
ml_data['Coerencia_Autoavaliacao'] = abs(ml_data['IAA'] - ml_data['IDA'])

features_para_modelo = ['IDA', 'IEG', 'IPS', 'IPP', 'IAA', 'IPV',
                        'Saude_Academica', 'Bem_Estar_Psico', 'Risco_Composto',
                        'Gap_Expectativa_Realidade', 'Coerencia_Autoavaliacao']

print(f"Features criadas: {len(features_para_modelo)}")
print(f"Registros para modelagem: {len(ml_data)}")
print(f"Distribuição: {ml_data['Risco_Defasagem'].value_counts().to_dict()}")

print("\nPreparando dados de treino e teste:")

X = ml_data[features_para_modelo]
y = ml_data['Risco_Defasagem']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Treino: {len(X_train)} registros")
print(f"Teste: {len(X_test)} registros")


Features criadas: 11
Registros para modelagem: 1985
Distribuição: {0: 1078, 1: 907}

Preparando dados de treino e teste:
Treino: 1588 registros
Teste: 397 registros


# Treinando Modelos

In [57]:
bundle = train_models(model_data)
bundle['model_name'], bundle['target_definition']

('Random Forest', 'Risco_Defasagem = 1 quando IAN < 7')

In [58]:
import pandas as pd

pd.DataFrame(bundle['metrics']).T[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']]

,accuracy,precision,recall,f1,roc_auc
Random Forest,0.599496,0.576389,0.458564,0.510769,0.61349
Gradient Boosting,0.59194,0.566434,0.447514,0.5,0.602747


In [59]:
import joblib

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, MODEL_PATH)
MODEL_PATH

PosixPath('/content/models/modelo_risco_defasagem.joblib')

## Random Forest

In [60]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

## Gradient Boosting

In [61]:
gb_model = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)
y_pred_proba_gb = gb_model.predict_proba(X_test)[:, 1]

In [62]:
print("\nAvaliando modelos:")

print("\n--- RANDOM FOREST ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"Precisão: {precision_score(y_test, y_pred_rf):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.3f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.3f}")

print("\n--- GRADIENT BOOSTING ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_gb):.3f}")
print(f"Precisão: {precision_score(y_test, y_pred_gb):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_gb):.3f}")
print(f"F1-Score: {f1_score(y_test, y_pred_gb):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_gb):.3f}")


Avaliando modelos:

--- RANDOM FOREST ---
Acurácia: 0.602
Precisão: 0.582
Recall: 0.453
F1-Score: 0.509
ROC-AUC: 0.611

--- GRADIENT BOOSTING ---
Acurácia: 0.582
Precisão: 0.548
Recall: 0.475
F1-Score: 0.509
ROC-AUC: 0.587


# Escolher melhor modelo

In [63]:
if roc_auc_score(y_test, y_pred_proba_rf) > roc_auc_score(y_test, y_pred_proba_gb):
    melhor_modelo = rf_model
    melhor_nome = "Random Forest"
    melhor_proba = y_pred_proba_rf
else:
    melhor_modelo = gb_model
    melhor_nome = "Gradient Boosting"
    melhor_proba = y_pred_proba_gb

print(f"\nMelhor modelo: {melhor_nome}")


Melhor modelo: Random Forest


# Aplicar modelo a todos os dados

In [64]:
X_all = ml_data[features_para_modelo]
probabilidades = melhor_modelo.predict_proba(X_all)[:, 1]
ml_data['Probabilidade_Risco'] = probabilidades
ml_data['Predicao_Risco'] = melhor_modelo.predict(X_all)

# Estatísticas finais

In [65]:
alto_risco = ml_data[ml_data['Probabilidade_Risco'] > 0.7]
risco_moderado = ml_data[(ml_data['Probabilidade_Risco'] > 0.5) & (ml_data['Probabilidade_Risco'] <= 0.7)]
sem_risco = ml_data[ml_data['Probabilidade_Risco'] <= 0.5]

print(f"\nESTATÍSTICAS FINAIS:")
print(f"Alunos em ALTO RISCO: {len(alto_risco)} ({len(alto_risco)/len(ml_data)*100:.1f}%)")
print(f"Alunos em RISCO MODERADO: {len(risco_moderado)} ({len(risco_moderado)/len(ml_data)*100:.1f}%)")
print(f"Alunos SEM RISCO: {len(sem_risco)} ({len(sem_risco)/len(ml_data)*100:.1f}%)")


ESTATÍSTICAS FINAIS:
Alunos em ALTO RISCO: 207 (10.4%)
Alunos em RISCO MODERADO: 619 (31.2%)
Alunos SEM RISCO: 1159 (58.4%)


# Modelo Preditivo de Risco de Defasagem

Notebook da entrega do Datathon FIAP para construcao de um modelo que estima a probabilidade de um aluno entrar em risco de defasagem educacional.

## Etapas exigidas

- Carregamento e limpeza da base 2022-2024
- Feature engineering
- Separacao treino/teste
- Modelagem preditiva
- Avaliacao dos resultados
- Exportacao do modelo para uso no Streamlit

In [66]:
from pathlib import Path
import sys

# Determine the project root based on the current working directory.
# This assumes 'src' is a direct subdirectory of the project root.
if Path.cwd().name == 'notebooks':
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

# Add the project root to the system path so Python can find modules within it.
sys.path.append(str(PROJECT_ROOT))

# Import from src.datathon_pipeline as the file is located in /content/src/
from src.datathon_pipeline import DATA_PATH, MODEL_PATH, add_features, load_dataset, train_models

DATA_PATH

PosixPath('/content/data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx')

In [67]:
data = load_dataset(DATA_PATH)
data.shape

(3030, 57)

In [68]:
data[['Ano', 'IDA', 'IEG', 'IPS', 'IPP', 'IAA', 'IPV', 'IAN']].describe()

,Ano,IDA,IEG,IPS,IPP,IAA,IPV,IAN
count,3030.00000,2852.000000,2954.000000,2859.000000,1992.000000,2865.000000,2852.000000,3030.000000
mean,2023.09769,6.375964,7.945696,6.287129,7.555203,7.918225,7.545476,7.179043
std,0.80995,1.956637,2.152281,1.792491,0.938990,2.626209,1.084347,2.535266
min,2022.00000,0.000000,0.000000,2.500000,2.500000,0.000000,2.500000,2.500000
25%,2022.00000,5.100000,7.300000,5.020000,7.083333,7.900000,6.984000,5.000000
50%,2023.00000,6.666667,8.600000,7.500000,7.500000,8.751000,7.583000,5.000000
75%,2024.00000,7.833333,9.400000,7.510000,8.125000,9.500000,8.255000,10.000000
max,2024.00000,10.000000,10.000000,10.000000,10.000000,10.002000,10.010000,10.000000


## Feature Engineering

O IAN foi tratado como indicador de adequação do nível: quanto menor o valor, maior a defasagem. Por isso, o alvo foi definido como `Risco_Defasagem = 1` quando `IAN < 7`.

Para evitar vazamento de informação, o IAN é usado apenas para criar o alvo de treino. Ele não entra como variável de entrada do modelo.

In [69]:
model_data = add_features(data)
model_data[['Categoria_IAN', 'Risco_Defasagem']].value_counts().sort_index()

,,count
Categoria_IAN,Risco_Defasagem,
Adequado,0,907
Levemente defasado,1,1062
Severamente defasado,1,16


In [70]:
model_data[
    ['IDA', 'IEG', 'IPS', 'IPP', 'IAA', 'IPV', 'IAN', 'Saude_Academica',
     'Bem_Estar_Psico', 'Risco_Composto', 'Gap_Expectativa_Realidade',
     'Coerencia_Autoavaliacao', 'Risco_Defasagem']
].head()

,IDA,IEG,IPS,IPP,IAA,IPV,IAN,Saude_Academica,Bem_Estar_Psico,Risco_Composto,Gap_Expectativa_Realidade,Coerencia_Autoavaliacao,Risco_Defasagem
860,9.6,10.0,8.13,8.4375,9.5,8.920,10.0,9.80,8.815,0,-0.680,0.1,0
861,8.9,9.1,8.14,7.5000,8.5,8.585,5.0,9.00,8.320,0,-0.315,0.4,1
862,6.3,7.6,3.14,5.9375,0.0,6.260,10.0,6.95,1.570,1,-0.040,6.3,0
863,6.3,7.6,8.14,7.5000,0.0,8.500,10.0,6.95,4.070,1,2.200,6.3,0
864,7.4,8.7,7.52,7.5000,8.5,7.915,10.0,8.05,8.010,0,0.515,1.1,0


# Variável alvo: Risco de Defasagem (IAN > 5)

In [71]:
model_data['Risco_Defasagem'] = (model_data['IAN'] > 5).astype(int)